# LSTM

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import mlflow
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import TensorDataset, DataLoader


In [2]:
train_df = pd.read_csv('../train_competition_2026.csv')
train_df['time'] = pd.to_datetime(train_df['time'], format='%Y-%m-%d %H:%M:%S')
train_df.head()

,obs,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4,y_1,y_2
0,0,0,2068-09-19 23:34:11,1.38,49,7,1,3,1,0,1,105.5,95.0,67.4,36.6,23.2,33.4,107.4
1,0,0,2068-09-19 23:35:11,1.38,49,7,1,3,1,0,1,104.4,95.0,66.4,37.8,22.7,33.4,107.4
2,0,0,2068-09-19 23:36:11,1.38,49,7,1,3,1,0,1,104.0,95.0,65.2,37.0,22.1,33.4,107.4
3,0,0,2068-09-19 23:37:11,1.38,49,7,1,3,1,0,1,102.8,95.0,63.4,35.9,20.7,33.4,107.4
4,0,0,2068-09-19 23:38:11,1.38,49,7,1,3,1,0,1,101.3,95.1,59.1,34.5,18.1,33.4,107.4


In [3]:
# Confirming every obs is 30 units long
counts = train_df['obs'].value_counts()
counts[counts != 30]

Series([], Name: count, dtype: int64)

In [4]:
test_df = pd.read_csv('../test_no_outcome.csv')
test_df['time'] = pd.to_datetime(test_df['time'], format='%Y-%m-%d %H:%M:%S')
test_df.head(3)

,obs,sub_id,time,num_0,num_1,num_2,cat_0,cat_1,cat_2,cat_3,cat_4,t_0,t_1,t_2,t_3,t_4
0,18,1,2134-04-01 22:23:14,-1.0,38,1,1,1,0,0,0,105.4,99.8,50.7,61.4,36.8
1,18,1,2134-04-01 22:24:14,-1.0,38,1,1,1,0,0,0,105.4,99.4,49.4,61.1,36.2
2,18,1,2134-04-01 22:25:14,-1.0,38,1,1,1,0,0,0,104.6,99.0,49.7,61.4,36.6


In [5]:
def prep_data(df, target_cols, drop_cols, seq_length=30):
    exclude = target_cols + drop_cols + ['obs']
    feature_cols = [col for col in df.columns if col not in exclude]
    
    X_list = []
    y_list = []
    
    for _, group in df.groupby('obs'):
        group = group.sort_values('time')
        
        features = group[feature_cols].values
        X_list.append(features)

        targets = group[target_cols].iloc[0].values
        y_list.append(targets)
        
    return np.stack(X_list), np.stack(y_list), feature_cols

In [6]:
def prep_data(df, target_cols, drop_cols, test_size=0.2):
    exclude = target_cols + drop_cols + ['obs']
    feature_cols = [col for col in df.columns if col not in exclude]
    
    continuous_cols = [col for col in feature_cols if col.startswith('num_') or col.startswith('t_')]
    
    unique_obs = df['obs'].unique()
    train_obs, test_obs = train_test_split(unique_obs, test_size=test_size, random_state=42)
    
    train_df = df[df['obs'].isin(train_obs)].copy()
    test_df = df[df['obs'].isin(test_obs)].copy()
    
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    
    if continuous_cols:
        train_df[continuous_cols] = x_scaler.fit_transform(train_df[continuous_cols])
        test_df[continuous_cols] = x_scaler.transform(test_df[continuous_cols])
        
    train_df[target_cols] = y_scaler.fit_transform(train_df[target_cols])
    test_df[target_cols] = y_scaler.transform(test_df[target_cols])
    
    def extract_sequences(data):
        X_list, y_list = [], []
        for _, group in data.groupby('obs'):
            group = group.sort_values('time')
            X_list.append(group[feature_cols].values)
            y_list.append(group[target_cols].iloc[0].values)
            
        return np.stack(X_list), np.stack(y_list)
        
    X_train, y_train = extract_sequences(train_df)
    X_test, y_test = extract_sequences(test_df)
    
    return X_train, y_train, X_test, y_test, y_scaler, feature_cols



In [7]:
target_columns = ['y_1', 'y_2']
columns_to_drop = ['time', 'sub_id'] 

X_train, y_train, X_test, y_test, target_scaler, features = prep_data(
    df=train_df, 
    target_cols=target_columns, 
    drop_cols=columns_to_drop
)

X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print(f"X shape: {X_tensor.shape}")
print(f"y shape: {y_tensor.shape}")

X shape: torch.Size([11536, 30, 13])
y shape: torch.Size([11536, 2])


In [8]:
class HealthSequenceDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)
        
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

dataset = HealthSequenceDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [9]:
class LSTM(nn.Module):
    def __init__(self, num_features, hidden_size, num_layers, num_targets, dropout):
        super(LSTM, self).__init__()
        
        dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=num_features, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout
        )
        
        self.fc = nn.Linear(hidden_size, num_targets)
        
    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        
        final_timestep_out = out[:, -1, :]
        
        predictions = self.fc(final_timestep_out)
        
        return predictions

In [10]:
def calculate_unscaled_rmse(preds, targets, scaler):
    unscaled_preds = scaler.inverse_transform(preds)
    unscaled_targets = scaler.inverse_transform(targets)
    
    rmses = []
    for i in range(unscaled_targets.shape[1]):
        rmse = root_mean_squared_error(unscaled_targets[:, i], unscaled_preds[:, i])
        rmses.append(rmse)
        
    return np.mean(rmses)

In [11]:
def train(X_tensor, y_tensor, X_test_tensor, y_test_tensor, hidden_size, num_layers, weight_decay, dropout, epochs=100, epochs_int=10):
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    val_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    model = LSTM(
        num_features=X_tensor.shape[2],
        hidden_size=hidden_size,
        num_layers=num_layers,
        num_targets=y_tensor.shape[1],
        dropout=dropout
    )

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)

    counter = 0
    patience = 40
    best_val_loss = float('inf')
    best_val_rmse = float('inf')
    best_model = None

    # Train
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        all_train_preds = []
        all_train_targets = []
        
        for batch_X, batch_y in dataloader:
            optimizer.zero_grad()
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            all_train_preds.append(outputs.detach().cpu().numpy())
            all_train_targets.append(batch_y.detach().cpu().numpy())
        
        train_preds_np = np.vstack(all_train_preds)
        train_targets_np = np.vstack(all_train_targets)
        
        train_rmse = calculate_unscaled_rmse(train_preds_np, train_targets_np, target_scaler)
        
        avg_loss = total_loss / len(dataloader)
        
        # Validate
        val_loss = 0.0
        val_preds, val_true = [], []

        with torch.no_grad():
            for xb, yb in val_loader:
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item()
                
                val_preds.append(pred)
                val_true.append(yb)

        val_loss /= len(val_loader)
        
        val_preds_tensor = torch.cat(val_preds)
        val_true_tensor = torch.cat(val_true)
        
        val_preds_np = val_preds_tensor.cpu().numpy()
        val_targets_np = val_true_tensor.cpu().numpy()
        
        val_rmse = calculate_unscaled_rmse(val_preds_np, val_targets_np, target_scaler)
        
        if val_loss < best_val_loss - 1e-4:
            best_model = model.state_dict()
            best_val_loss = val_loss
            best_val_rmse = val_rmse
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print("Early stopping triggered.")
            break

        if epoch % epochs_int == 0:
            print(f'Epoch {epoch} | Train RMSE {train_rmse:.4f} | Val RMSE {val_rmse:.4f}')
        
    return best_model, best_val_rmse

In [12]:
def run_pipeline(hp):    
    train_df = pd.read_csv('../train_competition_2026.csv')
    train_df['time'] = pd.to_datetime(train_df['time'], format='%Y-%m-%d %H:%M:%S')
    target_columns = ['y_1', 'y_2']
    columns_to_drop = ['time', 'sub_id'] 

    X_train, y_train, X_test, y_test, target_scaler, features = prep_data(
        df=train_df, 
        target_cols=target_columns, 
        drop_cols=columns_to_drop
    )

    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
    
    model, val_rmse = train(
        X_tensor, y_tensor, X_test_tensor, y_test_tensor,
        hp['hidden_size'],
        hp['num_layers'],
        hp['weight_decay'],
        hp['dropout'],
        epochs=100, epochs_int=10
        )
    
    return model, val_rmse

In [13]:
hp = {
    'hidden_size': 16,
    'num_layers': 2,
    'dropout': 0.0,
    'weight_decay': 1e-4,
    'epochs': 500, 
    'epoch_int': 1,
}

run_pipeline(hp)

Epoch 0 | Train RMSE 9.8632 | Val RMSE 7.1649
Epoch 10 | Train RMSE 6.6301 | Val RMSE 6.5942
Epoch 20 | Train RMSE 6.5226 | Val RMSE 6.5970
Epoch 30 | Train RMSE 6.4468 | Val RMSE 6.6123
Epoch 40 | Train RMSE 6.4087 | Val RMSE 6.6411
Epoch 50 | Train RMSE 6.3394 | Val RMSE 6.6475
Early stopping triggered.


(OrderedDict([('lstm.weight_ih_l0',
               tensor([[ 2.2045e-01,  1.5830e-01,  1.2029e-01,  2.8367e-02, -4.4830e-02,
                        -2.5950e-02, -8.7442e-03,  1.4324e-01,  3.3525e-01,  3.9625e-01,
                        -2.4903e-01,  1.0282e-01,  3.9952e-01],
                       [-7.0783e-02,  4.2138e-02, -4.6229e-02,  1.1264e-01,  8.9325e-02,
                         8.7940e-03,  8.5802e-04,  2.8385e-01,  3.8078e-02, -8.0370e-02,
                         2.1141e-01,  2.0622e-01,  2.1598e-01],
                       [ 1.8728e-01, -6.0673e-02, -6.7027e-02,  7.2155e-01,  1.7478e-01,
                         1.7447e-01,  7.5776e-03, -3.7515e-01,  4.2633e-01,  5.7366e-02,
                        -4.3633e-01, -4.3452e-03, -1.3979e-01],
                       [ 3.7412e-01,  6.5601e-01,  1.3710e-01,  5.1337e-01, -7.9262e-03,
                         1.8157e-01, -6.2501e-02, -1.1810e-02, -2.6483e-01, -1.0478e-01,
                         5.0339e-04,  2.1497e-01,  5.0802e-0

In [14]:
def run_with_mlflow(hp):
    run_name = f"lstm_h{hp['hidden_size']}_l{hp['num_layers']}_w{hp['weight_decay']}"
    with mlflow.start_run(run_name=run_name):
        # mlflow.set_tag("grid_search", "agg_stats_mlp")
        mlflow.log_params(hp)
        model, val_rmse = run_pipeline(hp)
        mlflow.log_metric("val_rmse", val_rmse)
        
        return model, val_rmse

if __name__ == '__main__':
    db_path = "/Users/connoryablonski/Dropbox/USF_MSDS/advanced_ml/mlflow.db"
    mlflow.set_tracking_uri(f"sqlite:///{db_path}")
    mlflow.set_experiment("AML_Project_LSTM_Grid_Search")
    
    print(f"Tracking URI: {mlflow.get_tracking_uri()}")
    print(f"Active Experiment: {mlflow.get_experiment_by_name('AML_Project_LSTM_Grid_Search').experiment_id}")

    from itertools import product

    best_score = float("inf")
    
    hidden_sizes = [32, 64, 128, 256]
    num_layers = [2, 4, 6, 8]
    dropouts = [0.0, 0.2, 0.4]
    weight_decays = [1e-5, 1e-4, 1e-3]
    
    best_model = None

    for hs, nl, dr, wd in product(hidden_sizes, num_layers, dropouts, weight_decays):
        
        hp = {
            'hidden_size': hs,
            'num_layers': nl,
            'dropout': dr,
            'weight_decay': wd,
        }
        
        model, score = run_with_mlflow(hp)
        
        if score < best_score:
            best_score = score
            best_hp = hp
            best_model = model

    print(f"Best config: {best_hp}, val_score: {best_score}")

Tracking URI: sqlite:////Users/connoryablonski/Dropbox/USF_MSDS/advanced_ml/mlflow.db
Active Experiment: 3
Epoch 0 | Train RMSE 8.5471 | Val RMSE 6.8777
Epoch 10 | Train RMSE 6.5866 | Val RMSE 6.5803
Epoch 20 | Train RMSE 6.4210 | Val RMSE 6.5579
Epoch 30 | Train RMSE 6.2815 | Val RMSE 6.6497
Epoch 40 | Train RMSE 6.1541 | Val RMSE 6.7070
Epoch 50 | Train RMSE 5.9771 | Val RMSE 6.8094
Early stopping triggered.
Epoch 0 | Train RMSE 8.4618 | Val RMSE 6.8653
Epoch 10 | Train RMSE 6.5852 | Val RMSE 6.6845
Epoch 20 | Train RMSE 6.4718 | Val RMSE 6.5980
Epoch 30 | Train RMSE 6.3511 | Val RMSE 6.6807
Epoch 40 | Train RMSE 6.2553 | Val RMSE 6.6681
Early stopping triggered.
Epoch 0 | Train RMSE 8.6396 | Val RMSE 7.0050
Epoch 10 | Train RMSE 6.6802 | Val RMSE 6.6557
Epoch 20 | Train RMSE 6.6250 | Val RMSE 6.5926
Epoch 30 | Train RMSE 6.5763 | Val RMSE 6.5513
Epoch 40 | Train RMSE 6.5582 | Val RMSE 6.5763


KeyboardInterrupt: 